# Notebook 23 — LOSO Susceptibility Analysis with Raw/ERP/Power and ICA Features + FDR

This notebook extends the subject-level susceptibility analysis to the LOSO setting.

The key difference from the within-subject susceptibility notebooks is interpretive:

- Within-subject susceptibility asks: which subjects are easy to classify when training and testing within the same subject?
- LOSO susceptibility asks: which held-out subjects are easy to generalize to when the model is trained on other subjects?

Therefore, in this notebook, subject features always describe the **held-out test subject**, not the training subjects.

The notebook uses the existing model set only:

- LDA
- Logistic Regression
- EEGNet

No new neural models are added here.

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)

from oddball_eeg.config import get_experiment
from oddball_eeg.visualization import round_numeric_columns


ARTIFACT_ROOT = PROJECT_ROOT / "results" / "thesis_artifacts"

OUT_DIR = ARTIFACT_ROOT / "loso_susceptibility_fdr"
TABLE_DIR = OUT_DIR / "tables"
FIGURE_DIR = OUT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output directory:", OUT_DIR)

In [ ]:
# These are the selected high-subject-count wide-timebin experiments.
# ds003061 has only 13 available subjects, so its max30 config still uses 13.

MAIN_EXPERIMENT_NAMES = [
    "ds003061_within_subject_max30_wide_timebin",
    "ds005863_within_subject_max30_wide_timebin",
    "ds006466_within_subject_max30_wide_timebin",
    "ds006480_within_subject_max30_wide_timebin",
]

LOSO_EXPERIMENT_NAMES = [
    name.replace("_within_subject_", "_loso_")
    for name in MAIN_EXPERIMENT_NAMES
]

MODEL_ORDER = ["lda", "logreg", "eegnet"]

print("LOSO experiments:")
for name in LOSO_EXPERIMENT_NAMES:
    print("-", name)

In [ ]:
def save_dataframe(df: pd.DataFrame, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Saved: {path}")


def bh_fdr(p_values):
    """
    Benjamini-Hochberg FDR correction.

    Returns q-values in the original order.
    Works without requiring statsmodels.
    """
    p = np.asarray(p_values, dtype=float)
    q = np.full_like(p, np.nan, dtype=float)

    valid = np.isfinite(p)
    if valid.sum() == 0:
        return q

    p_valid = p[valid]
    n = len(p_valid)

    order = np.argsort(p_valid)
    ranked = p_valid[order]

    q_ranked = ranked * n / (np.arange(1, n + 1))
    q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
    q_ranked = np.minimum(q_ranked, 1.0)

    q_valid = np.empty_like(q_ranked)
    q_valid[order] = q_ranked

    q[valid] = q_valid
    return q


def safe_spearman(x, y):
    """
    Spearman correlation with safety checks.
    """
    x = pd.Series(x)
    y = pd.Series(y)

    mask = x.notna() & y.notna()
    x = x[mask]
    y = y[mask]

    if len(x) < 5:
        return np.nan, np.nan, int(len(x))

    if x.nunique(dropna=True) < 3 or y.nunique(dropna=True) < 3:
        return np.nan, np.nan, int(len(x))

    rho, p = stats.spearmanr(x, y)
    return float(rho), float(p), int(len(x))


def zscore_within_group(series: pd.Series) -> pd.Series:
    """
    Z-score within a dataset group.
    Uses ddof=0 to avoid instability in small groups.
    """
    s = pd.to_numeric(series, errors="coerce")
    mu = s.mean()
    sd = s.std(ddof=0)

    if not np.isfinite(sd) or sd == 0:
        return pd.Series(np.nan, index=series.index)

    return (s - mu) / sd


def normalize_model_name(model):
    m = str(model).strip().lower()

    aliases = {
        "logistic_regression": "logreg",
        "logistic regression": "logreg",
        "lr": "logreg",
        "lda": "lda",
        "eegnet": "eegnet",
        "current_eegnet": "eegnet",
    }

    return aliases.get(m, m)


def make_dataset_label(row):
    dataset = str(row.get("dataset", "")).strip()
    event_preset = str(row.get("event_preset", "")).strip()

    if "dataset_label" in row and pd.notna(row.get("dataset_label")):
        existing = str(row.get("dataset_label")).strip()
        if existing and existing.lower() != "nan":
            return existing

    if dataset == "ds006480":
        if event_preset and event_preset.lower() not in ["nan", "none", ""]:
            return f"{dataset}_{event_preset}"
        return "ds006480_pooled"

    return dataset


def normalize_subject_id(subject, dataset_label=None, dataset=None):
    """
    Normalize subject IDs for stable merging.

    ds003061 and ds005863 commonly use 3-digit numeric IDs.
    ds006480 commonly uses 2-digit numeric IDs.
    ds006466 uses non-numeric names.
    """
    if pd.isna(subject):
        return np.nan

    s = str(subject).strip()

    if s.startswith("sub-"):
        s = s.replace("sub-", "", 1)

    if s.endswith(".0"):
        s = s[:-2]

    label = str(dataset_label or dataset or "")

    if s.isdigit():
        if label.startswith("ds003061") or label.startswith("ds005863"):
            return s.zfill(3)
        if label.startswith("ds006480"):
            return s.zfill(2)

    return s


def feature_family_from_name(feature_name: str) -> str:
    f = str(feature_name).lower()

    if any(k in f for k in ["p300", "erp", "target_standard", "target_minus_standard", "cohen", "snr"]):
        return "erp_p300_separability"

    if any(k in f for k in ["trial", "n_target", "n_standard", "pos_rate", "class_balance"]):
        return "trial_count_balance"

    if any(k in f for k in ["baseline", "noise", "rms", "std"]):
        return "baseline_noise"

    if any(k in f for k in ["theta", "alpha", "beta", "gamma", "delta", "band", "power", "ratio", "spectral"]):
        return "spectral_power"

    if any(k in f for k in ["ica", "ic_", "kurtosis", "entropy", "mixing", "unmixing", "roughness", "smoothness", "condition", "mutual"]):
        return "ica_source_quality"

    return "other"

In [ ]:
def find_loso_subject_file(cfg):
    """
    Find the clean LOSO subject AUC file if available.
    Otherwise fall back to the older subject summary file.
    """
    base = Path(cfg.result_dir())

    preferred_names = [
        "loso_subject_auc.csv",
        "loso_subject_summary_across_seeds.csv",
    ]

    for filename in preferred_names:
        matches = sorted(base.rglob(filename))
        if matches:
            return matches[0]

    return None


loso_file_rows = []

for loso_name in LOSO_EXPERIMENT_NAMES:
    cfg = get_experiment(loso_name)
    found = find_loso_subject_file(cfg)

    loso_file_rows.append({
        "loso_experiment": loso_name,
        "dataset": cfg.dataset.dataset,
        "task": cfg.dataset.task,
        "event_preset": cfg.dataset.event_preset,
        "run_tag": cfg.run_tag,
        "result_dir": str(cfg.result_dir()),
        "status": "found" if found is not None else "missing",
        "path": str(found) if found is not None else "missing",
    })

loso_file_status_df = pd.DataFrame(loso_file_rows)

save_dataframe(
    loso_file_status_df,
    TABLE_DIR / "loso_file_status.csv",
)

display(loso_file_status_df)

In [ ]:
def load_one_loso_subject_auc(row):
    if row["status"] != "found":
        return None

    path = Path(row["path"])
    df = pd.read_csv(path)

    cfg = get_experiment(row["loso_experiment"])

    # Add/repair dataset metadata
    df["loso_experiment"] = row["loso_experiment"]
    df["dataset"] = cfg.dataset.dataset
    df["task"] = cfg.dataset.task
    df["event_preset"] = cfg.dataset.event_preset
    df["run_tag"] = cfg.run_tag

    if "dataset_label" not in df.columns:
        df["dataset_label"] = df.apply(make_dataset_label, axis=1)
    else:
        df["dataset_label"] = df.apply(make_dataset_label, axis=1)

    # Normalize subject column
    if "heldout_subject" not in df.columns:
        if "subject" in df.columns:
            df["heldout_subject"] = df["subject"]
        else:
            raise ValueError(f"No subject or heldout_subject column in {path}")

    df["heldout_subject"] = [
        normalize_subject_id(s, dataset_label=dl, dataset=ds)
        for s, dl, ds in zip(df["heldout_subject"], df["dataset_label"], df["dataset"])
    ]

    if "subject" not in df.columns:
        df["subject"] = df["heldout_subject"]

    df["subject"] = [
        normalize_subject_id(s, dataset_label=dl, dataset=ds)
        for s, dl, ds in zip(df["subject"], df["dataset_label"], df["dataset"])
    ]

    # Normalize model names
    if "model" not in df.columns:
        raise ValueError(f"No model column in {path}")

    df["model"] = df["model"].map(normalize_model_name)

    # Normalize AUC column
    if "loso_auc" not in df.columns:
        candidate_auc_cols = [
            "auc_seed_mean",
            "auc_mean",
            "mean_auc",
            "auc",
        ]

        found_auc = None
        for col in candidate_auc_cols:
            if col in df.columns:
                found_auc = col
                break

        if found_auc is None:
            raise ValueError(
                f"Could not find LOSO AUC column in {path}. "
                f"Available columns: {list(df.columns)}"
            )

        df["loso_auc"] = df[found_auc]

    if "loso_auc_std" not in df.columns:
        candidate_std_cols = [
            "auc_seed_std",
            "auc_std",
            "std_auc",
        ]

        found_std = None
        for col in candidate_std_cols:
            if col in df.columns:
                found_std = col
                break

        df["loso_auc_std"] = df[found_std] if found_std is not None else np.nan

    # Keep only the current intended model set
    df = df[df["model"].isin(MODEL_ORDER)].copy()

    # Add feature metadata if missing
    if "feature_mode" not in df.columns:
        df["feature_mode"] = np.where(df["model"].isin(["lda", "logreg"]), "time_binned", "raw_epoch")

    if "bin_ms" not in df.columns:
        df["bin_ms"] = np.where(df["model"].isin(["lda", "logreg"]), 50, np.nan)

    if "crop_tmin" not in df.columns:
        df["crop_tmin"] = 0.0

    if "crop_tmax" not in df.columns:
        df["crop_tmax"] = 0.8

    keep_cols = [
        "dataset_label",
        "dataset",
        "task",
        "event_preset",
        "heldout_subject",
        "subject",
        "model",
        "loso_auc",
        "loso_auc_std",
        "feature_mode",
        "bin_ms",
        "crop_tmin",
        "crop_tmax",
        "run_tag",
        "loso_experiment",
    ]

    keep_cols = [c for c in keep_cols if c in df.columns]
    return df[keep_cols].copy()


loso_dfs = []

for _, row in loso_file_status_df.iterrows():
    loaded = load_one_loso_subject_auc(row)
    if loaded is not None:
        loso_dfs.append(loaded)

if not loso_dfs:
    raise RuntimeError(
        "No LOSO subject AUC files were loaded. "
        "Run Notebook 04 for the max30 wide-timebin LOSO configs first."
    )

loso_subject_auc_df = pd.concat(loso_dfs, ignore_index=True)

loso_subject_auc_df["loso_auc"] = pd.to_numeric(
    loso_subject_auc_df["loso_auc"],
    errors="coerce",
)

loso_subject_auc_df = (
    loso_subject_auc_df
    .dropna(subset=["dataset_label", "heldout_subject", "model", "loso_auc"])
    .drop_duplicates(subset=["dataset_label", "heldout_subject", "model"], keep="last")
    .sort_values(["dataset_label", "model", "heldout_subject"])
    .reset_index(drop=True)
)

save_dataframe(
    loso_subject_auc_df,
    TABLE_DIR / "loso_subject_auc.csv",
)

print("LOSO subject AUC shape:", loso_subject_auc_df.shape)
display(round_numeric_columns(loso_subject_auc_df.head(20), 4))

In [ ]:
loso_model_summary_df = (
    loso_subject_auc_df
    .groupby(["dataset_label", "model"], as_index=False)
    .agg(
        n_subjects=("heldout_subject", "nunique"),
        loso_mean_auc=("loso_auc", "mean"),
        loso_std_auc=("loso_auc", "std"),
        loso_median_auc=("loso_auc", "median"),
        loso_min_auc=("loso_auc", "min"),
        loso_max_auc=("loso_auc", "max"),
    )
)

loso_model_summary_df["model"] = pd.Categorical(
    loso_model_summary_df["model"],
    categories=MODEL_ORDER,
    ordered=True,
)

loso_model_summary_df = (
    loso_model_summary_df
    .sort_values(["dataset_label", "model"])
    .reset_index(drop=True)
)

save_dataframe(
    loso_model_summary_df,
    TABLE_DIR / "loso_model_summary_from_subject_auc.csv",
)

display(round_numeric_columns(loso_model_summary_df, 4))

## Load Raw/ERP/Power Subject Features

The raw/ERP/power feature table is expected to come from Notebook 17 or 17b.

This table should have one row per subject-recording and contain features such as:

- baseline RMS / noise
- ROI baseline noise
- P300-window target-standard difference
- ERP SNR
- single-trial P300 separability
- spectral power features
- trial-count features

For LOSO, these features are merged using:

`dataset_label + heldout_subject`

In [ ]:
def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


raw_feature_candidates = [
    ARTIFACT_ROOT / "classification_susceptibility" / "tables" / "subject_raw_eeg_features.csv",
    ARTIFACT_ROOT / "classification_susceptibility" / "tables" / "subject_raw_erp_power_features.csv",
    ARTIFACT_ROOT / "multimodel_classification_susceptibility" / "tables" / "subject_raw_eeg_features.csv",
    ARTIFACT_ROOT / "multimodel_classification_susceptibility" / "tables" / "subject_raw_erp_power_features.csv",
]

raw_feature_path = find_first_existing(raw_feature_candidates)

if raw_feature_path is None:
    matches = sorted((PROJECT_ROOT / "results").rglob("subject_raw*features*.csv"))
    if matches:
        raw_feature_path = matches[0]

if raw_feature_path is None:
    raise FileNotFoundError(
        "Could not find raw/ERP/power subject feature table. "
        "Run Notebook 17 or 17b first."
    )

raw_features_df = pd.read_csv(raw_feature_path)

print("Raw/ERP/power feature table:", raw_feature_path)
print("Shape:", raw_features_df.shape)
print("Columns:", len(raw_features_df.columns))

display(raw_features_df.head())

In [ ]:
def normalize_subject_feature_table(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    out = df.copy()

    if "dataset_label" not in out.columns:
        if "dataset" not in out.columns:
            raise ValueError(f"{table_name} has no dataset or dataset_label column.")
        out["dataset_label"] = out.apply(make_dataset_label, axis=1)
    else:
        out["dataset_label"] = out.apply(make_dataset_label, axis=1)

    if "subject" not in out.columns:
        candidate_subject_cols = [
            "heldout_subject",
            "subject_id",
            "sub",
        ]

        found_subject = None
        for col in candidate_subject_cols:
            if col in out.columns:
                found_subject = col
                break

        if found_subject is None:
            raise ValueError(f"{table_name} has no subject column.")

        out["subject"] = out[found_subject]

    dataset_col = out["dataset"] if "dataset" in out.columns else out["dataset_label"]

    out["subject"] = [
        normalize_subject_id(s, dataset_label=dl, dataset=ds)
        for s, dl, ds in zip(out["subject"], out["dataset_label"], dataset_col)
    ]

    return out


raw_features_df = normalize_subject_feature_table(
    raw_features_df,
    table_name="raw_features_df",
)

raw_features_df = (
    raw_features_df
    .drop_duplicates(subset=["dataset_label", "subject"], keep="last")
    .sort_values(["dataset_label", "subject"])
    .reset_index(drop=True)
)

save_dataframe(
    raw_features_df,
    TABLE_DIR / "subject_raw_erp_power_features_normalized.csv",
)

print("Normalized raw/ERP/power features:", raw_features_df.shape)
display(raw_features_df[["dataset_label", "subject"]].head(20))

## Load ICA Subject Features

The ICA feature table is expected to come from Notebook 18.

This table should have one row per subject-recording and contain source-quality features such as:

- IC kurtosis
- IC spectral entropy
- IC correlation / mutual information
- spatial roughness / smoothness
- component map sparsity
- mixing and unmixing matrix statistics

For LOSO, these features are also merged using:

`dataset_label + heldout_subject`

In [ ]:
ica_feature_candidates = [
    ARTIFACT_ROOT / "ica_multimodel_classification_susceptibility" / "tables" / "subject_ica_features.csv",
    ARTIFACT_ROOT / "ica_multimodel_classification_susceptibility" / "tables" / "ica_subject_features.csv",
]

ica_feature_path = find_first_existing(ica_feature_candidates)

if ica_feature_path is None:
    matches = sorted((PROJECT_ROOT / "results").rglob("*ica*subject*features*.csv"))
    if matches:
        ica_feature_path = matches[0]

if ica_feature_path is None:
    raise FileNotFoundError(
        "Could not find ICA subject feature table. "
        "Run Notebook 18 with RECOMPUTE_ICA_FEATURES = True first."
    )

ica_features_df = pd.read_csv(ica_feature_path)

print("ICA feature table:", ica_feature_path)
print("Shape:", ica_features_df.shape)
print("Columns:", len(ica_features_df.columns))

display(ica_features_df.head())

In [ ]:
ica_features_df = normalize_subject_feature_table(
    ica_features_df,
    table_name="ica_features_df",
)

ica_features_df = (
    ica_features_df
    .drop_duplicates(subset=["dataset_label", "subject"], keep="last")
    .sort_values(["dataset_label", "subject"])
    .reset_index(drop=True)
)

save_dataframe(
    ica_features_df,
    TABLE_DIR / "subject_ica_features_normalized.csv",
)

print("Normalized ICA features:", ica_features_df.shape)
display(ica_features_df[["dataset_label", "subject"]].head(20))

In [ ]:
loso_raw_merged_df = loso_subject_auc_df.merge(
    raw_features_df,
    left_on=["dataset_label", "heldout_subject"],
    right_on=["dataset_label", "subject"],
    how="left",
    suffixes=("", "_feature"),
)

raw_merge_status_df = (
    loso_raw_merged_df
    .assign(has_raw_features=lambda d: d["subject_feature"].notna() if "subject_feature" in d.columns else d["subject"].notna())
    .groupby(["dataset_label", "model"], as_index=False)
    .agg(
        n_loso_rows=("heldout_subject", "count"),
        n_subjects=("heldout_subject", "nunique"),
        n_with_features=("has_raw_features", "sum"),
    )
)

save_dataframe(
    loso_raw_merged_df,
    TABLE_DIR / "loso_raw_feature_merged_subject_table.csv",
)

save_dataframe(
    raw_merge_status_df,
    TABLE_DIR / "loso_raw_feature_merge_status.csv",
)

print("Merged LOSO + raw/ERP/power features:", loso_raw_merged_df.shape)
display(raw_merge_status_df)

In [ ]:
loso_ica_merged_df = loso_subject_auc_df.merge(
    ica_features_df,
    left_on=["dataset_label", "heldout_subject"],
    right_on=["dataset_label", "subject"],
    how="left",
    suffixes=("", "_feature"),
)

ica_merge_status_df = (
    loso_ica_merged_df
    .assign(has_ica_features=lambda d: d["subject_feature"].notna() if "subject_feature" in d.columns else d["subject"].notna())
    .groupby(["dataset_label", "model"], as_index=False)
    .agg(
        n_loso_rows=("heldout_subject", "count"),
        n_subjects=("heldout_subject", "nunique"),
        n_with_features=("has_ica_features", "sum"),
    )
)

save_dataframe(
    loso_ica_merged_df,
    TABLE_DIR / "loso_ica_feature_merged_subject_table.csv",
)

save_dataframe(
    ica_merge_status_df,
    TABLE_DIR / "loso_ica_feature_merge_status.csv",
)

print("Merged LOSO + ICA features:", loso_ica_merged_df.shape)
display(ica_merge_status_df)

In [ ]:
IDENTIFIER_AND_TARGET_COLS = {
    "dataset",
    "dataset_label",
    "task",
    "event_preset",
    "subject",
    "subject_feature",
    "heldout_subject",
    "model",
    "loso_auc",
    "loso_auc_std",
    "feature_mode",
    "bin_ms",
    "crop_tmin",
    "crop_tmax",
    "run_tag",
    "loso_experiment",
    "experiment_name",
}


def get_numeric_feature_columns(df: pd.DataFrame) -> list[str]:
    numeric_cols = []

    for col in df.columns:
        if col in IDENTIFIER_AND_TARGET_COLS:
            continue

        if col.endswith("_filename") or col.endswith("_path"):
            continue

        values = pd.to_numeric(df[col], errors="coerce")

        if values.notna().sum() >= 5 and values.nunique(dropna=True) >= 3:
            numeric_cols.append(col)

    return numeric_cols


raw_feature_cols = get_numeric_feature_columns(loso_raw_merged_df)
ica_feature_cols = get_numeric_feature_columns(loso_ica_merged_df)

print("Raw/ERP/power feature columns:", len(raw_feature_cols))
print(raw_feature_cols[:40])

print("\nICA feature columns:", len(ica_feature_cols))
print(ica_feature_cols[:40])

In [ ]:
def compute_within_dataset_correlations(
        merged_df: pd.DataFrame,
        feature_cols: list[str],
        feature_source: str,
) -> pd.DataFrame:
    rows = []

    for model in MODEL_ORDER:
        model_df = merged_df[merged_df["model"] == model].copy()

        for dataset_label, ds_df in model_df.groupby("dataset_label"):
            for feature in feature_cols:
                x = pd.to_numeric(ds_df[feature], errors="coerce")
                y = pd.to_numeric(ds_df["loso_auc"], errors="coerce")

                rho, p, n = safe_spearman(x, y)

                rows.append({
                    "feature_source": feature_source,
                    "correlation_scope": "within_dataset",
                    "dataset_label": dataset_label,
                    "model": model,
                    "feature": feature,
                    "feature_family": feature_family_from_name(feature),
                    "n_subjects": n,
                    "spearman_rho": rho,
                    "p_value": p,
                    "abs_rho": abs(rho) if np.isfinite(rho) else np.nan,
                })

    out = pd.DataFrame(rows)

    if len(out):
        out["q_value_within_source"] = bh_fdr(out["p_value"].values)
        out["significant_q05"] = out["q_value_within_source"] <= 0.05
        out["suggestive_q10"] = out["q_value_within_source"] <= 0.10

    return out


def compute_grand_zscored_correlations(
        merged_df: pd.DataFrame,
        feature_cols: list[str],
        feature_source: str,
) -> pd.DataFrame:
    rows = []

    for model in MODEL_ORDER:
        model_df = merged_df[merged_df["model"] == model].copy()

        for feature in feature_cols:
            temp = model_df[[
                "dataset_label",
                "heldout_subject",
                "loso_auc",
                feature,
            ]].copy()

            temp[feature] = pd.to_numeric(temp[feature], errors="coerce")
            temp["loso_auc"] = pd.to_numeric(temp["loso_auc"], errors="coerce")

            temp = temp.dropna(subset=[feature, "loso_auc"])

            if temp.empty:
                rows.append({
                    "feature_source": feature_source,
                    "correlation_scope": "grand_zscored",
                    "model": model,
                    "feature": feature,
                    "feature_family": feature_family_from_name(feature),
                    "n_subjects": 0,
                    "n_datasets": 0,
                    "spearman_rho": np.nan,
                    "p_value": np.nan,
                    "abs_rho": np.nan,
                })
                continue

            temp["feature_z"] = temp.groupby("dataset_label")[feature].transform(zscore_within_group)
            temp["loso_auc_z"] = temp.groupby("dataset_label")["loso_auc"].transform(zscore_within_group)

            temp = temp.dropna(subset=["feature_z", "loso_auc_z"])

            rho, p, n = safe_spearman(temp["feature_z"], temp["loso_auc_z"])

            rows.append({
                "feature_source": feature_source,
                "correlation_scope": "grand_zscored",
                "model": model,
                "feature": feature,
                "feature_family": feature_family_from_name(feature),
                "n_subjects": n,
                "n_datasets": temp["dataset_label"].nunique(),
                "spearman_rho": rho,
                "p_value": p,
                "abs_rho": abs(rho) if np.isfinite(rho) else np.nan,
            })

    out = pd.DataFrame(rows)

    if len(out):
        out["q_value_within_source"] = bh_fdr(out["p_value"].values)
        out["significant_q05"] = out["q_value_within_source"] <= 0.05
        out["suggestive_q10"] = out["q_value_within_source"] <= 0.10

    return out

## Raw/ERP/Power LOSO Susceptibility Correlations

This section asks whether held-out subject LOSO AUC is associated with raw signal quality, ERP/P300 separability, power features, or trial-count properties.

The grand analysis uses dataset-wise z-scoring so that cross-dataset scale differences do not dominate the correlation.

In [ ]:
loso_raw_within_dataset_corr_df = compute_within_dataset_correlations(
    merged_df=loso_raw_merged_df,
    feature_cols=raw_feature_cols,
    feature_source="raw_erp_power",
)

loso_raw_grand_corr_df = compute_grand_zscored_correlations(
    merged_df=loso_raw_merged_df,
    feature_cols=raw_feature_cols,
    feature_source="raw_erp_power",
)

save_dataframe(
    loso_raw_within_dataset_corr_df,
    TABLE_DIR / "loso_raw_within_dataset_correlations.csv",
)

save_dataframe(
    loso_raw_grand_corr_df,
    TABLE_DIR / "loso_raw_grand_correlations.csv",
)

save_dataframe(
    loso_raw_grand_corr_df,
    TABLE_DIR / "loso_raw_grand_correlations_fdr.csv",
)

print("Top raw/ERP/power grand correlations:")
display(
    round_numeric_columns(
        loso_raw_grand_corr_df
        .sort_values(["q_value_within_source", "p_value", "abs_rho"], ascending=[True, True, False])
        .head(30),
        4,
    )
)

## ICA LOSO Susceptibility Correlations

This section asks whether held-out subject LOSO AUC is associated with ICA/source-quality features.

These results should be interpreted cautiously because previous within-subject ICA susceptibility effects were exploratory and did not survive grand FDR correction.

In [ ]:
loso_ica_within_dataset_corr_df = compute_within_dataset_correlations(
    merged_df=loso_ica_merged_df,
    feature_cols=ica_feature_cols,
    feature_source="ica",
)

loso_ica_grand_corr_df = compute_grand_zscored_correlations(
    merged_df=loso_ica_merged_df,
    feature_cols=ica_feature_cols,
    feature_source="ica",
)

save_dataframe(
    loso_ica_within_dataset_corr_df,
    TABLE_DIR / "loso_ica_within_dataset_correlations.csv",
)

save_dataframe(
    loso_ica_grand_corr_df,
    TABLE_DIR / "loso_ica_grand_correlations.csv",
)

save_dataframe(
    loso_ica_grand_corr_df,
    TABLE_DIR / "loso_ica_grand_correlations_fdr.csv",
)

print("Top ICA grand correlations:")
display(
    round_numeric_columns(
        loso_ica_grand_corr_df
        .sort_values(["q_value_within_source", "p_value", "abs_rho"], ascending=[True, True, False])
        .head(30),
        4,
    )
)

In [ ]:
combined_grand_corr_df = pd.concat(
    [
        loso_raw_grand_corr_df,
        loso_ica_grand_corr_df,
    ],
    ignore_index=True,
)

# Global FDR across both feature sources.
combined_grand_corr_df["q_value_global"] = bh_fdr(combined_grand_corr_df["p_value"].values)
combined_grand_corr_df["significant_global_q05"] = combined_grand_corr_df["q_value_global"] <= 0.05
combined_grand_corr_df["suggestive_global_q10"] = combined_grand_corr_df["q_value_global"] <= 0.10

combined_grand_corr_df = (
    combined_grand_corr_df
    .sort_values(["q_value_global", "p_value", "abs_rho"], ascending=[True, True, False])
    .reset_index(drop=True)
)

loso_spearman_fdr05_results_df = combined_grand_corr_df[
    combined_grand_corr_df["significant_global_q05"]
].copy()

loso_spearman_fdr10_results_df = combined_grand_corr_df[
    combined_grand_corr_df["suggestive_global_q10"]
].copy()

save_dataframe(
    combined_grand_corr_df,
    TABLE_DIR / "loso_all_grand_correlations_global_fdr.csv",
)

save_dataframe(
    loso_spearman_fdr05_results_df,
    TABLE_DIR / "loso_spearman_fdr05_results.csv",
)

save_dataframe(
    loso_spearman_fdr10_results_df,
    TABLE_DIR / "loso_spearman_fdr10_results.csv",
)

print("Global q <= 0.05 results:", len(loso_spearman_fdr05_results_df))
display(round_numeric_columns(loso_spearman_fdr05_results_df, 4))

print("Global q <= 0.10 exploratory results:", len(loso_spearman_fdr10_results_df))
display(round_numeric_columns(loso_spearman_fdr10_results_df.head(50), 4))

In [ ]:
loso_fdr_summary_by_family_df = (
    combined_grand_corr_df
    .groupby(["feature_source", "feature_family", "model"], as_index=False)
    .agg(
        n_tests=("feature", "count"),
        n_q05_source=("significant_q05", "sum"),
        n_q10_source=("suggestive_q10", "sum"),
        n_q05_global=("significant_global_q05", "sum"),
        n_q10_global=("suggestive_global_q10", "sum"),
        best_abs_rho=("abs_rho", "max"),
        best_p_value=("p_value", "min"),
        best_q_source=("q_value_within_source", "min"),
        best_q_global=("q_value_global", "min"),
    )
)

loso_fdr_summary_by_family_df = (
    loso_fdr_summary_by_family_df
    .sort_values(["best_q_global", "best_p_value", "feature_source", "feature_family", "model"])
    .reset_index(drop=True)
)

save_dataframe(
    loso_fdr_summary_by_family_df,
    TABLE_DIR / "loso_fdr_summary_by_family.csv",
)

display(round_numeric_columns(loso_fdr_summary_by_family_df, 4))

In [ ]:
top_loso_effects_by_model_df = (
    combined_grand_corr_df
    .sort_values(["model", "q_value_global", "p_value", "abs_rho"], ascending=[True, True, True, False])
    .groupby("model", as_index=False)
    .head(15)
    .reset_index(drop=True)
)

save_dataframe(
    top_loso_effects_by_model_df,
    TABLE_DIR / "loso_top_grand_effects_by_model.csv",
)

display(round_numeric_columns(top_loso_effects_by_model_df, 4))

In [ ]:
combined_within_dataset_corr_df = pd.concat(
    [
        loso_raw_within_dataset_corr_df,
        loso_ica_within_dataset_corr_df,
    ],
    ignore_index=True,
)

combined_within_dataset_corr_df["q_value_global_within_dataset_table"] = bh_fdr(
    combined_within_dataset_corr_df["p_value"].values
)

combined_within_dataset_corr_df["significant_global_q05"] = (
        combined_within_dataset_corr_df["q_value_global_within_dataset_table"] <= 0.05
)

combined_within_dataset_corr_df["suggestive_global_q10"] = (
        combined_within_dataset_corr_df["q_value_global_within_dataset_table"] <= 0.10
)

combined_within_dataset_corr_df = (
    combined_within_dataset_corr_df
    .sort_values(
        ["q_value_global_within_dataset_table", "p_value", "abs_rho"],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

save_dataframe(
    combined_within_dataset_corr_df,
    TABLE_DIR / "loso_all_within_dataset_correlations_global_fdr.csv",
)

top_within_dataset_loso_effects_df = (
    combined_within_dataset_corr_df
    .sort_values(
        ["dataset_label", "model", "q_value_global_within_dataset_table", "p_value", "abs_rho"],
        ascending=[True, True, True, True, False],
    )
    .groupby(["dataset_label", "model"], as_index=False)
    .head(10)
    .reset_index(drop=True)
)

save_dataframe(
    top_within_dataset_loso_effects_df,
    TABLE_DIR / "loso_top_within_dataset_effects_by_dataset_model.csv",
)

display(round_numeric_columns(top_within_dataset_loso_effects_df, 4))

In [ ]:
grand_interpretation_table_df = combined_grand_corr_df[[
    "feature_source",
    "feature_family",
    "model",
    "feature",
    "n_subjects",
    "n_datasets",
    "spearman_rho",
    "p_value",
    "q_value_within_source",
    "q_value_global",
    "significant_global_q05",
    "suggestive_global_q10",
]].copy()

grand_interpretation_table_df = (
    grand_interpretation_table_df
    .sort_values(["q_value_global", "p_value", "abs_rho"] if "abs_rho" in grand_interpretation_table_df.columns else ["q_value_global", "p_value"])
    .reset_index(drop=True)
)

save_dataframe(
    grand_interpretation_table_df,
    TABLE_DIR / "loso_grand_interpretation_table.csv",
)

display(round_numeric_columns(grand_interpretation_table_df.head(40), 4))

In [ ]:
import matplotlib.pyplot as plt

plot_df = loso_subject_auc_df.copy()

model_labels = {
    "lda": "LDA",
    "logreg": "LogReg",
    "eegnet": "EEGNet",
}

plot_df["model_label"] = plot_df["model"].map(model_labels).fillna(plot_df["model"])

summary_for_plot = (
    plot_df
    .groupby(["dataset_label", "model_label"], as_index=False)
    .agg(
        mean_auc=("loso_auc", "mean"),
        std_auc=("loso_auc", "std"),
    )
)

pivot = summary_for_plot.pivot(
    index="dataset_label",
    columns="model_label",
    values="mean_auc",
)

ax = pivot.plot(kind="bar", figsize=(10, 5))
ax.set_title("LOSO Mean ROC-AUC by Dataset and Model")
ax.set_xlabel("Dataset")
ax.set_ylabel("Mean LOSO ROC-AUC")
ax.set_ylim(0.4, 1.0)
ax.legend(title="Model")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

plot_path = FIGURE_DIR / "loso_mean_auc_by_dataset_model.png"
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", plot_path)

## Final Notes for Interpretation

Use caution when interpreting this notebook.

A significant or suggestive LOSO susceptibility effect means that a held-out subject characteristic is associated with how well models trained on other subjects generalize to that subject.

This is not identical to within-subject susceptibility.

Recommended wording:

- LOSO susceptibility reflects cross-subject generalization difficulty.
- The subject features describe the held-out test subject.
- ERP/P300 separability remains the most physiologically interpretable susceptibility family if it appears among the strongest effects.
- ICA/source-quality effects should remain exploratory unless they survive FDR correction clearly.
- If no grand effects survive FDR, report that cross-dataset LOSO susceptibility trends remain exploratory under multiple-comparison correction.

In [ ]:
output_files = sorted(TABLE_DIR.glob("*.csv"))

inventory_df = pd.DataFrame({
    "file": [p.name for p in output_files],
    "path": [str(p) for p in output_files],
})

save_dataframe(
    inventory_df,
    TABLE_DIR / "loso_susceptibility_output_inventory.csv",
)

display(inventory_df)

In [ ]:
print("Notebook 23 completed.")
print("Main outputs saved in:", TABLE_DIR)
print()
print("Most important tables:")
print("-", TABLE_DIR / "loso_subject_auc.csv")
print("-", TABLE_DIR / "loso_raw_feature_merged_subject_table.csv")
print("-", TABLE_DIR / "loso_ica_feature_merged_subject_table.csv")
print("-", TABLE_DIR / "loso_raw_grand_correlations_fdr.csv")
print("-", TABLE_DIR / "loso_ica_grand_correlations_fdr.csv")
print("-", TABLE_DIR / "loso_spearman_fdr05_results.csv")
print("-", TABLE_DIR / "loso_spearman_fdr10_results.csv")
print("-", TABLE_DIR / "loso_fdr_summary_by_family.csv")